# Kaggle × Colab runner

Clone the code pushed from local to GitHub and run it on Colab.

**First time only:** register the following in the 🔑 Secrets sidebar and enable "Notebook access".

| Name | Value |
|---|---|
| `KAGGLE_API_TOKEN` | Kaggle > Settings > API > *Generate New Token* |
| (legacy) `KAGGLE_USERNAME` / `KAGGLE_KEY` | contents of `kaggle.json` |
| `GITHUB_TOKEN` | only for private repos (fine-grained PAT, Contents: Read-only) |

In [ ]:
# ===== Settings =====
GITHUB_USER = "Moriyuki-S"
REPO_NAME   = "kaggle_ws"
BRANCH      = "main"

COMPETITION = "titanic"       # Kaggle competition slug
EXP         = "exp001"        # runs competitions/<COMPETITION>/exp/<EXP>/train.py
SCRIPT_ARGS = ""              # e.g. "--debug"

USE_DRIVE   = True            # keep outputs (models, submission) on Google Drive

In [ ]:
# ===== Drive mount & secrets =====
import os

from google.colab import userdata

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def _secret(name):
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None

for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
    if (v := _secret(k)):
        os.environ[k] = v
assert os.environ.get("KAGGLE_API_TOKEN") or os.environ.get("KAGGLE_KEY"), "Register Kaggle credentials in Colab Secrets"
GITHUB_TOKEN = _secret("GITHUB_TOKEN")

In [ ]:
# ===== Clone / pull the repo =====
REPO_DIR = f"/content/{REPO_NAME}"
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
url = f"https://{auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --oneline

In [ ]:
# ===== Install =====
# --no-deps: install only our package and keep Colab's preinstalled libraries intact
!pip install -q -U kaggle
!pip install -q --no-deps -e .
# extra packages: !pip install -q xxx

In [ ]:
# ===== Download data (skipped if present; re-run after runtime reset) =====
# Accept the competition rules on Kaggle first, otherwise you get 403
DATA_DIR = f"/content/data/{COMPETITION}"
if not os.path.exists(DATA_DIR):
    !bash scripts/download.sh {COMPETITION} {DATA_DIR}
!ls -lh {DATA_DIR}

In [ ]:
# ===== Run =====
!nvidia-smi -L || true
!python competitions/{COMPETITION}/exp/{EXP}/train.py {SCRIPT_ARGS}

In [ ]:
# ===== Submit (run only when needed) =====
from kaggle_ws import get_paths

sub = get_paths(COMPETITION).output / EXP / "submission.csv"
!kaggle competitions submit -c {COMPETITION} -f "{sub}" -m "{EXP}"
!kaggle competitions submissions -c {COMPETITION} | head -n 5